In [ ]:
#import libraries
import pandas as pd
import getpass
from sqlalchemy import create_engine, text

In [ ]:
#configure database connection 
db_user = "postgres"
db_password = getpass.getpass("PostgreSQL password: ")
db_host = "localhost"
db_port = "5432"
db_name = "ai_data_analyst"

database_url = (
    f"postgresql+psycopg2://{db_user}:{db_password}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(database_url)

In [ ]:
#testing database connection 
with engine.connect() as connection:
    database_name = connection.execute(
        text("SELECT current_database();")
    ).scalar()

print("Connected database:", database_name)

In [ ]:
#query total orders
query = """
SELECT COUNT(*) AS total_orders
FROM orders;
"""

total_orders = pd.read_sql(query, engine)

total_orders

In [ ]:
#query revenue overview
query = """
SELECT
    ROUND(SUM(price), 2) AS total_revenue,
    ROUND(AVG(price), 2) AS average_item_price,
    COUNT(*) AS total_items_sold
FROM order_items;
"""

revenue_overview = pd.read_sql(query, engine)

revenue_overview

In [ ]:
#query monthly revenue
query = """
SELECT
    TO_CHAR(o.order_purchase_timestamp, 'YYYY-MM') AS month,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items AS oi
JOIN orders AS o
    ON oi.order_id = o.order_id
GROUP BY
    TO_CHAR(o.order_purchase_timestamp, 'YYYY-MM')
ORDER BY month;
"""

monthly_revenue_sql = pd.read_sql(query, engine)

monthly_revenue_sql


In [ ]:
#query top categories by revenue
query = """
SELECT
    ct.product_category_name_english AS category,
    ROUND(SUM(oi.price), 2) AS revenue
FROM order_items AS oi
JOIN products AS p
    ON oi.product_id = p.product_id
LEFT JOIN category_translation AS ct
    ON p.product_category_name = ct.product_category_name
GROUP BY ct.product_category_name_english
ORDER BY revenue DESC
LIMIT 10;
"""

top_categories_sql = pd.read_sql(query, engine)

top_categories_sql

In [ ]:
#query customer order behaviour
query = """
WITH customer_orders AS (
    SELECT
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count
    FROM customers AS c
    JOIN orders AS o
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
)
SELECT
    COUNT(*) FILTER (WHERE order_count = 1) AS one_time_customers,
    COUNT(*) FILTER (WHERE order_count > 1) AS repeat_customers,
    ROUND(
        100.0 * COUNT(*) FILTER (WHERE order_count > 1) / COUNT(*),
        2
    ) AS repeat_customer_rate,
    MAX(order_count) AS maximum_orders
FROM customer_orders;
"""

customer_behavior_sql = pd.read_sql(query, engine)

customer_behavior_sql

In [ ]:
#query delivery impact on reviews
query = """
SELECT
    CASE
        WHEN DATE_PART(
            'day',
            o.order_delivered_customer_date - o.order_estimated_delivery_date
        ) > 0
            THEN 'Late'
        ELSE 'On time or early'
    END AS delivery_status,
    COUNT(*) AS review_count,
    ROUND(AVG(r.review_score), 2) AS average_review_score
FROM orders AS o
JOIN reviews AS r
    ON o.order_id = r.order_id
WHERE o.order_delivered_customer_date IS NOT NULL
GROUP BY delivery_status
ORDER BY delivery_status;
"""

delivery_reviews_sql = pd.read_sql(query, engine)

delivery_reviews_sql

In [ ]:
#validate reference queries
validation_results = {
    "Total orders": int(total_orders.iloc[0]["total_orders"]) == 99441,
    "Total revenue": float(revenue_overview.iloc[0]["total_revenue"]) == 13591643.70,
    "Total items sold": int(revenue_overview.iloc[0]["total_items_sold"]) == 112650,
    "Repeat customer rate": float(customer_behavior_sql.iloc[0]["repeat_customer_rate"]) == 3.12,
    "Late delivery review score": float(
        delivery_reviews_sql.loc[
            delivery_reviews_sql["delivery_status"] == "Late",
            "average_review_score"
        ].iloc[0]
    ) == 2.27
}

for metric, passed in validation_results.items():
    print(f"{metric}: {'PASS' if passed else 'FAIL'}")